# Koemi-3HIP A100 — nine-hour code and reasoning run

This notebook is self-contained for Colab. Run the cells in order. The first code cell pins the repository commit and installs only the streaming dataset dependency. The second cell embeds the tested A100 runner and executes its mocked data, lazy-causal-dataset and rotating-checkpoint tests before any remote dataset is opened.

The training cell requires an NVIDIA A100 with at least 70 GiB visible VRAM. It runs one nine-hour training session, checkpoints to two rotating Drive slots every 15 minutes, and can be run again to resume the exact epoch permutation and optimizer state. It uses the current HERM parallel path, BF16 autocast and TF32 matrix math. Accelerate and torch.compile are intentionally not enabled without a measured equivalence and throughput gate.

The corpus is English-first and code-heavy: OpenCodeInstruct, CodeFeedback, Magicoder and verified OpenR1-Math traces. OpenCode rows require a perfect recorded test score and all test statuses to pass. Terminal-Bench and BigCodeBench are evaluation-only and are never executed or copied into the training corpus.

The final cell reads the report and generated sample after the nine-hour cell completes. A Colab provider can still terminate a session; the latest Drive checkpoint is the recovery boundary.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Koemi-AI/Koemi-3HIP.git'
REPOSITORY_REVISION = '94ab988145889279599ae12299b31a508847cbbd'
REPOSITORY_DIR = Path('/content/Koemi-3HIP')
if not (REPOSITORY_DIR / 'src' / 'koemi').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run(
    ['git', '-C', str(REPOSITORY_DIR), 'fetch', '--quiet', 'origin', REPOSITORY_REVISION],
    check=True,
)
subprocess.run(
    ['git', '-C', str(REPOSITORY_DIR), 'checkout', '--quiet', '--detach', REPOSITORY_REVISION],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR / 'src'))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'datasets==3.6.0'],
    check=True,
)

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive', force_remount=False)
    RESULTS_DIR = Path('/content/drive/MyDrive/koemi-3hip-a100-code-reasoning-v1')
else:
    RESULTS_DIR = Path('/content/koemi-3hip-a100-code-reasoning-v1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print({'repository_revision': REPOSITORY_REVISION, 'results_dir': str(RESULTS_DIR)})

In [ ]:
import json

A100_RUN_SOURCE = "from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport pickle\nimport random\nimport tempfile\nimport time\nimport uuid\nfrom array import array\nfrom collections import Counter\nfrom collections.abc import Callable, Iterator, Mapping, Sequence\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor\nfrom torch.utils.data import DataLoader, Dataset, Sampler\n\nfrom koemi.configuration.settings import ModelSettings, PAD_TOKEN_ID\nfrom koemi.data.contracts import DatasetRecord, DatasetValidationError\nfrom koemi.data.serialization import serialize_record\nfrom koemi.data.tokenizer import ByteTokenizer\nfrom koemi.model.execution import ExecutionMode\nfrom koemi.model.network import KoemiModel\nfrom koemi.training.checkpoints import CheckpointStore\nfrom koemi.training.dataset import CausalByteDataset, IGNORE_TARGET_ID\nfrom koemi.training.generation import generate_text\nfrom koemi.training.objective import calculate_training_objective, token_cross_entropy\n\n\nRUN_FORMAT_VERSION = 1\nCORPUS_FORMAT_VERSION = 1\nCHECKPOINT_FORMAT_VERSION = 1\nMINIMUM_A100_MEMORY_BYTES = 70 * 2**30\nCODE_SYSTEM_PROMPT = (\n    \"You are a precise English software engineer. Diagnose errors, explain the cause, \"\n    \"provide a minimal safe change or command, and state how to verify it.\"\n)\nMATH_SYSTEM_PROMPT = (\n    \"Solve in English. Keep the algebra explicit, check each inference, and present \"\n    \"the final answer separately.\"\n)\nPRIORITY_CODE_TERMS = (\n    \"python\",\n    \"typescript\",\n    \"javascript\",\n    \"node\",\n    \"npm\",\n    \"bash\",\n    \"powershell\",\n    \"shell\",\n    \"terminal\",\n    \"command line\",\n    \"cli\",\n    \"git\",\n    \"commit\",\n    \"rebase\",\n    \"merge\",\n    \"debug\",\n    \"bug\",\n    \"error\",\n    \"exception\",\n    \"traceback\",\n    \"test\",\n    \"pytest\",\n)\nSOURCE_REVISIONS = {\n    \"opencode\": \"8f3ba5bafe4d6e8db46082cf7ae6741bc370604d\",\n    \"codefeedback\": \"a08c213a9748c66c15d0225814be80a2e77adf4a\",\n    \"magicoder\": \"b0079beaa0361d82412520b873715bee59cc7dd4\",\n    \"openr1_math\": \"e4e141ec9dea9f8326f4d347be56105859b2bd68\",\n}\n\n\nclass SourceRowRejected(DatasetValidationError):\n    pass\n\n\n@dataclass(frozen=True)\nclass CorpusQuotas:\n    opencode_priority: int\n    opencode_general: int\n    codefeedback: int\n    magicoder: int\n    openr1_math: int\n\n    def to_dict(self) -> dict[str, int]:\n        return asdict(self)\n\n\n@dataclass(frozen=True)\nclass RunConfiguration:\n    results_directory: Path\n    session_seconds: int\n    data_seed: int\n    model_seed: int\n    sequence_length: int\n    quotas: CorpusQuotas\n    opencode_scan_limit: int\n    source_scan_limit: int\n    shuffle_buffer_size: int\n    num_workers: int\n    checkpoint_interval_seconds: int\n    log_interval_steps: int\n    evaluation_batches: int\n\n\n@dataclass(frozen=True)\nclass CheckpointLoadResult:\n    payload: dict[str, Any] | None\n    recovery_messages: tuple[str, ...]\n\n\nclass MaterializedCausalByteDataset(Dataset[tuple[bytes, bytes, bytes, bytes]]):\n    def __init__(self, records: Sequence[DatasetRecord], sequence_length: int) -> None:\n        if sequence_length < 2:\n            raise ValueError(\"sequence_length must be at least 2\")\n        self.sequence_length = sequence_length\n        self.token_streams: list[bytes] = []\n        self.supervised_streams: list[bytes] = []\n        self.thinking_streams: list[bytes] = []\n        self.record_indices = array(\"I\")\n        self.offsets = array(\"I\")\n        supervised_token_count = 0\n        for record_index, record in enumerate(records):\n            serialized = serialize_record(record)\n            token_stream = serialized.token_bytes\n            if len(token_stream) < 2:\n                continue\n            supervised_stream = bytes(serialized.supervised_positions)\n            thinking_stream = bytes(serialized.thinking_positions)\n            storage_index = len(self.token_streams)\n            self.token_streams.append(token_stream)\n            self.supervised_streams.append(supervised_stream)\n            self.thinking_streams.append(thinking_stream)\n            for offset in range(0, len(token_stream) - 1, sequence_length):\n                end_offset = min(offset + sequence_length, len(token_stream) - 1)\n                chunk_supervised_token_count = sum(supervised_stream[offset + 1 : end_offset + 1])\n                if chunk_supervised_token_count == 0:\n                    continue\n                self.record_indices.append(storage_index)\n                self.offsets.append(offset)\n                supervised_token_count += chunk_supervised_token_count\n        if not self.record_indices:\n            raise DatasetValidationError(\"dataset does not contain a trainable causal sequence\")\n        if supervised_token_count == 0:\n            raise DatasetValidationError(\"dataset does not contain supervised target tokens\")\n        self.source_record_count = len(records)\n\n    def __len__(self) -> int:\n        return len(self.record_indices)\n\n    def __getitem__(self, index: int) -> tuple[bytes, bytes, bytes, bytes]:\n        stream_index = self.record_indices[index]\n        offset = self.offsets[index]\n        token_stream = self.token_streams[stream_index]\n        end_offset = min(offset + self.sequence_length, len(token_stream) - 1)\n        return (\n            token_stream[offset:end_offset],\n            token_stream[offset + 1 : end_offset + 1],\n            self.supervised_streams[stream_index][offset + 1 : end_offset + 1],\n            self.thinking_streams[stream_index][offset + 1 : end_offset + 1],\n        )\n\n\ndef collate_materialized_chunks(chunks: list[tuple[bytes, bytes, bytes, bytes]]) -> dict[str, Tensor]:\n    if not chunks:\n        raise ValueError(\"cannot collate an empty batch\")\n    maximum_length = max(len(input_bytes) for input_bytes, _, _, _ in chunks)\n    input_ids = torch.full((len(chunks), maximum_length), PAD_TOKEN_ID, dtype=torch.long)\n    target_ids = torch.full((len(chunks), maximum_length), IGNORE_TARGET_ID, dtype=torch.long)\n    thinking_mask = torch.zeros((len(chunks), maximum_length), dtype=torch.bool)\n    for row_index, (input_bytes, target_bytes, supervised_bytes, thinking_bytes) in enumerate(chunks):\n        length = len(input_bytes)\n        input_ids[row_index, :length] = torch.tensor(bytearray(input_bytes), dtype=torch.long)\n        target_values = torch.tensor(bytearray(target_bytes), dtype=torch.long)\n        supervised_values = torch.tensor(bytearray(supervised_bytes), dtype=torch.bool)\n        target_values = torch.where(\n            supervised_values,\n            target_values,\n            torch.full_like(target_values, IGNORE_TARGET_ID),\n        )\n        target_ids[row_index, :length] = target_values\n        thinking_mask[row_index, :length] = torch.tensor(bytearray(thinking_bytes), dtype=torch.bool)\n    return {\"input_ids\": input_ids, \"target_ids\": target_ids, \"thinking_mask\": thinking_mask}\n\n\nclass DeterministicBatchSampler(Sampler[list[int]]):\n    def __init__(\n        self,\n        dataset_length: int,\n        batch_size: int,\n        data_seed: int,\n        epoch_index: int,\n        start_batch_index: int,\n    ) -> None:\n        if dataset_length < 1:\n            raise ValueError(\"dataset_length must be positive\")\n        if batch_size < 1:\n            raise ValueError(\"batch_size must be positive\")\n        self.dataset_length = dataset_length\n        self.batch_size = batch_size\n        self.data_seed = data_seed\n        self.epoch_index = epoch_index\n        self.start_batch_index = start_batch_index\n        self.batch_count = math.ceil(dataset_length / batch_size)\n        if not 0 <= start_batch_index <= self.batch_count:\n            raise ValueError(\"start_batch_index is outside this epoch\")\n\n    def __iter__(self) -> Iterator[list[int]]:\n        generator = torch.Generator().manual_seed(self.data_seed + self.epoch_index)\n        order = torch.randperm(self.dataset_length, generator=generator).tolist()\n        for batch_index in range(self.start_batch_index, self.batch_count):\n            start_offset = batch_index * self.batch_size\n            yield order[start_offset : start_offset + self.batch_size]\n\n    def __len__(self) -> int:\n        return self.batch_count - self.start_batch_index\n\n\nclass RotatingCheckpointStore:\n    def __init__(self, directory: Path, run_signature: str) -> None:\n        self.directory = directory\n        self.run_signature = run_signature\n        self.manifest_path = directory / \"checkpoint_manifest.json\"\n\n    @property\n    def slot_paths(self) -> tuple[Path, Path]:\n        return self.directory / \"checkpoint_a.pt\", self.directory / \"checkpoint_b.pt\"\n\n    def save(self, payload: dict[str, Any], optimizer_step: int) -> Path:\n        if optimizer_step < 0:\n            raise ValueError(\"optimizer_step must be non-negative\")\n        self.directory.mkdir(parents=True, exist_ok=True)\n        slot_path = self.slot_paths[optimizer_step % len(self.slot_paths)]\n        saved_payload = dict(payload)\n        saved_payload[\"checkpoint_format_version\"] = CHECKPOINT_FORMAT_VERSION\n        saved_payload[\"run_signature\"] = self.run_signature\n        saved_payload[\"optimizer_step\"] = optimizer_step\n        atomic_torch_save(slot_path, saved_payload)\n        atomic_write_json(\n            self.manifest_path,\n            {\n                \"checkpoint_format_version\": CHECKPOINT_FORMAT_VERSION,\n                \"run_signature\": self.run_signature,\n                \"active_slot\": slot_path.name,\n                \"optimizer_step\": optimizer_step,\n            },\n        )\n        return slot_path\n\n    def load_latest(self) -> CheckpointLoadResult:\n        recovery_messages: list[str] = []\n        if self.manifest_path.exists():\n            manifest = read_json_object(self.manifest_path)\n            if manifest.get(\"run_signature\") != self.run_signature:\n                raise ValueError(\"checkpoint manifest belongs to a different run signature\")\n            active_name = manifest.get(\"active_slot\")\n            if not isinstance(active_name, str):\n                raise ValueError(\"checkpoint manifest has no active slot\")\n            candidates = [self.directory / active_name]\n            candidates.extend(path for path in self.slot_paths if path not in candidates)\n        else:\n            candidates = list(self.slot_paths)\n        valid_payloads: list[tuple[int, dict[str, Any]]] = []\n        for path in candidates:\n            if not path.exists():\n                continue\n            try:\n                payload = torch.load(path, map_location=\"cpu\", weights_only=True)\n                optimizer_step = validate_checkpoint_payload(payload, self.run_signature)\n                valid_payloads.append((optimizer_step, payload))\n            except (EOFError, OSError, pickle.UnpicklingError, RuntimeError, ValueError) as error:\n                recovery_messages.append(f\"ignored invalid checkpoint {path.name}: {type(error).__name__}: {error}\")\n        if valid_payloads:\n            optimizer_step, payload = max(valid_payloads, key=lambda item: item[0])\n            if recovery_messages:\n                recovery_messages.append(f\"resumed checkpoint at optimizer step {optimizer_step}\")\n            return CheckpointLoadResult(payload, tuple(recovery_messages))\n        if recovery_messages:\n            raise RuntimeError(\"no valid checkpoint slot remains: \" + \" | \".join(recovery_messages))\n        return CheckpointLoadResult(None, ())\n\n\nclass RunningMetrics:\n    def __init__(self, device: torch.device, expert_count: int) -> None:\n        self.device = device\n        self.loss_sum = torch.zeros((), device=device, dtype=torch.float64)\n        self.answer_loss_sum = torch.zeros((), device=device, dtype=torch.float64)\n        self.thinking_loss_sum = torch.zeros((), device=device, dtype=torch.float64)\n        self.loss_square_sum = torch.zeros((), device=device, dtype=torch.float64)\n        self.supervised_count = torch.zeros((), device=device, dtype=torch.long)\n        self.answer_count = torch.zeros((), device=device, dtype=torch.long)\n        self.thinking_count = torch.zeros((), device=device, dtype=torch.long)\n        self.expert_counts = torch.zeros(expert_count, device=device, dtype=torch.long)\n        self.valid_token_count = 0\n\n    def add(\n        self,\n        output,\n        token_losses: Tensor,\n        target_ids: Tensor,\n        thinking_mask: Tensor,\n    ) -> None:\n        supervised_mask = target_ids != IGNORE_TARGET_ID\n        answer_mask = supervised_mask & ~thinking_mask\n        thinking_positions = supervised_mask & thinking_mask\n        self.loss_sum += (token_losses * supervised_mask).sum().detach().to(torch.float64)\n        self.answer_loss_sum += (token_losses * answer_mask).sum().detach().to(torch.float64)\n        self.thinking_loss_sum += (token_losses * thinking_positions).sum().detach().to(torch.float64)\n        self.loss_square_sum += (token_losses.square() * supervised_mask).sum().detach().to(torch.float64)\n        self.supervised_count += supervised_mask.sum().detach()\n        self.answer_count += answer_mask.sum().detach()\n        self.thinking_count += thinking_positions.sum().detach()\n        self.valid_token_count += output.token_count\n        if output.active_expert_indices is not None:\n            assignments = output.active_expert_indices[output.valid_positions].reshape(-1)\n            self.expert_counts += torch.bincount(assignments, minlength=len(self.expert_counts))\n\n    def as_dict(self, elapsed_seconds: float) -> dict[str, Any]:\n        supervised_count = int(self.supervised_count.item())\n        answer_count = int(self.answer_count.item())\n        thinking_count = int(self.thinking_count.item())\n        if supervised_count == 0:\n            raise RuntimeError(\"metrics contain no supervised targets\")\n        loss_sum = float(self.loss_sum.item())\n        answer_loss_sum = float(self.answer_loss_sum.item())\n        thinking_loss_sum = float(self.thinking_loss_sum.item())\n        mean_loss = loss_sum / supervised_count\n        variance = max(0.0, float(self.loss_square_sum.item()) / supervised_count - mean_loss * mean_loss)\n        expert_counts = self.expert_counts.cpu().tolist()\n        return {\n            \"loss_nats\": mean_loss,\n            \"bpb\": mean_loss / math.log(2),\n            \"perplexity\": math.exp(min(mean_loss, 80.0)),\n            \"answer_loss_nats\": answer_loss_sum / answer_count if answer_count else None,\n            \"answer_bpb\": answer_loss_sum / answer_count / math.log(2) if answer_count else None,\n            \"thinking_loss_nats\": thinking_loss_sum / thinking_count if thinking_count else None,\n            \"thinking_bpb\": thinking_loss_sum / thinking_count / math.log(2) if thinking_count else None,\n            \"loss_standard_error\": math.sqrt(variance / supervised_count),\n            \"supervised_tokens\": supervised_count,\n            \"answer_tokens\": answer_count,\n            \"thinking_tokens\": thinking_count,\n            \"valid_input_tokens\": self.valid_token_count,\n            \"supervised_tokens_per_second\": supervised_count / max(elapsed_seconds, 1e-9),\n            \"expert_load\": expert_load_summary(expert_counts),\n        }\n\n\ndef atomic_write_text(path: Path, text: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path = path.with_name(f\".{path.name}.{uuid.uuid4().hex}.tmp\")\n    try:\n        temporary_path.write_text(text, encoding=\"utf-8\")\n        os.replace(temporary_path, path)\n    except BaseException:\n        if temporary_path.exists():\n            temporary_path.unlink()\n        raise\n\n\ndef atomic_write_json(path: Path, value: Mapping[str, Any]) -> None:\n    atomic_write_text(path, json.dumps(value, indent=2, sort_keys=True) + \"\\n\")\n\n\ndef atomic_torch_save(path: Path, payload: dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path = path.with_name(f\".{path.name}.{uuid.uuid4().hex}.tmp\")\n    try:\n        torch.save(payload, temporary_path)\n        os.replace(temporary_path, path)\n    except BaseException:\n        if temporary_path.exists():\n            temporary_path.unlink()\n        raise\n\n\ndef read_json_object(path: Path) -> dict[str, Any]:\n    value = json.loads(path.read_text(encoding=\"utf-8\"))\n    if not isinstance(value, dict):\n        raise ValueError(f\"{path} must contain a JSON object\")\n    return value\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as source_file:\n        for block in iter(lambda: source_file.read(1024 * 1024), b\"\"):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef fingerprint(value: Mapping[str, Any]) -> str:\n    encoded = json.dumps(value, sort_keys=True, separators=(\",\", \":\")).encode(\"utf-8\")\n    return hashlib.sha256(encoded).hexdigest()\n\n\ndef validate_checkpoint_payload(payload: Any, run_signature: str) -> int:\n    if not isinstance(payload, dict):\n        raise ValueError(\"checkpoint payload is not a dictionary\")\n    if payload.get(\"checkpoint_format_version\") != CHECKPOINT_FORMAT_VERSION:\n        raise ValueError(\"checkpoint format version does not match\")\n    if payload.get(\"run_signature\") != run_signature:\n        raise ValueError(\"checkpoint run signature does not match\")\n    optimizer_step = payload.get(\"optimizer_step\")\n    if not isinstance(optimizer_step, int) or optimizer_step < 0:\n        raise ValueError(\"checkpoint optimizer step is invalid\")\n    return optimizer_step\n\n\ndef require_text(raw_value: Any, field_name: str) -> str:\n    if not isinstance(raw_value, str) or not raw_value.strip():\n        raise SourceRowRejected(f\"field '{field_name}' must be a non-empty string\")\n    return raw_value.strip()\n\n\ndef require_bounded_text(raw_value: Any, field_name: str, maximum_bytes: int) -> str:\n    value = require_text(raw_value, field_name)\n    if len(value.encode(\"utf-8\")) > maximum_bytes:\n        raise SourceRowRejected(f\"field '{field_name}' exceeds {maximum_bytes} UTF-8 bytes\")\n    return value\n\n\ndef parse_full_test_score(raw_value: Any) -> None:\n    if not isinstance(raw_value, str):\n        raise SourceRowRejected(\"average_test_score must be a string\")\n    try:\n        score = float(raw_value)\n    except ValueError as error:\n        raise SourceRowRejected(\"average_test_score is not a number\") from error\n    if score != 1.0:\n        raise SourceRowRejected(\"average_test_score is not exactly 1.0\")\n\n\ndef require_all_test_statuses_pass(raw_value: Any) -> None:\n    if not isinstance(raw_value, str):\n        raise SourceRowRejected(\"tests_execution_status must be a JSON string\")\n    try:\n        statuses = json.loads(raw_value)\n    except json.JSONDecodeError as error:\n        raise SourceRowRejected(\"tests_execution_status is not valid JSON\") from error\n    if not isinstance(statuses, list) or not statuses or any(status != \"pass\" for status in statuses):\n        raise SourceRowRejected(\"tests_execution_status contains a non-passing test\")\n\n\ndef bounded_record(\n    identifier: str,\n    input_text: str,\n    output_text: str,\n    metadata: dict[str, Any],\n    system_text: str,\n    thinking_text: str | None = None,\n) -> DatasetRecord:\n    if len(input_text.encode(\"utf-8\")) > 8_192:\n        raise SourceRowRejected(\"input exceeds 8192 UTF-8 bytes\")\n    if len(output_text.encode(\"utf-8\")) > 12_288:\n        raise SourceRowRejected(\"output exceeds 12288 UTF-8 bytes\")\n    if thinking_text is not None and len(thinking_text.encode(\"utf-8\")) > 12_288:\n        raise SourceRowRejected(\"thinking exceeds 12288 UTF-8 bytes\")\n    return DatasetRecord(identifier, input_text, thinking_text, output_text, metadata, system_text)\n\n\ndef adapt_opencode_row(raw_row: Mapping[str, Any], fallback_identifier: str) -> DatasetRecord:\n    parse_full_test_score(raw_row.get(\"average_test_score\"))\n    require_all_test_statuses_pass(raw_row.get(\"tests_execution_status\"))\n    identifier = require_text(raw_row.get(\"id\", fallback_identifier), \"id\")\n    input_text = require_bounded_text(raw_row.get(\"input\"), \"input\", 8_192)\n    output_text = require_bounded_text(raw_row.get(\"output\"), \"output\", 12_288)\n    domain = require_text(raw_row.get(\"domain\", \"unknown\"), \"domain\")\n    return bounded_record(\n        f\"opencode:{identifier}\",\n        input_text,\n        output_text,\n        {\"source\": \"nvidia/OpenCodeInstruct\", \"revision\": SOURCE_REVISIONS[\"opencode\"], \"domain\": domain},\n        CODE_SYSTEM_PROMPT,\n    )\n\n\ndef adapt_codefeedback_row(raw_row: Mapping[str, Any], fallback_identifier: str) -> DatasetRecord:\n    query = require_bounded_text(raw_row.get(\"query\"), \"query\", 8_192)\n    answer = require_bounded_text(raw_row.get(\"answer\"), \"answer\", 12_288)\n    language = require_text(raw_row.get(\"lang\", \"unknown\"), \"lang\")\n    return bounded_record(\n        f\"codefeedback:{fallback_identifier}\",\n        query,\n        answer,\n        {\"source\": \"m-a-p/CodeFeedback-Filtered-Instruction\", \"revision\": SOURCE_REVISIONS[\"codefeedback\"], \"language\": language},\n        CODE_SYSTEM_PROMPT,\n    )\n\n\ndef adapt_magicoder_row(raw_row: Mapping[str, Any], fallback_identifier: str) -> DatasetRecord:\n    instruction = require_bounded_text(raw_row.get(\"instruction\"), \"instruction\", 8_192)\n    response = require_bounded_text(raw_row.get(\"response\"), \"response\", 12_288)\n    return bounded_record(\n        f\"magicoder:{fallback_identifier}\",\n        instruction,\n        response,\n        {\"source\": \"ise-uiuc/Magicoder-Evol-Instruct-110K\", \"revision\": SOURCE_REVISIONS[\"magicoder\"]},\n        CODE_SYSTEM_PROMPT,\n    )\n\n\ndef extract_reasoning_span(generation: str) -> str:\n    stripped = generation.strip()\n    if \"<think>\" not in stripped:\n        return stripped\n    opening = stripped.find(\"<think>\") + len(\"<think>\")\n    closing = stripped.find(\"</think>\", opening)\n    if closing < 0:\n        raise SourceRowRejected(\"verified math generation has an unclosed think span\")\n    reasoning = stripped[opening:closing].strip()\n    if not reasoning:\n        raise SourceRowRejected(\"verified math generation has an empty think span\")\n    return reasoning\n\n\ndef adapt_openr1_math_row(raw_row: Mapping[str, Any], fallback_identifier: str) -> DatasetRecord:\n    problem = require_bounded_text(raw_row.get(\"problem\"), \"problem\", 8_192)\n    answer = require_bounded_text(raw_row.get(\"answer\"), \"answer\", 12_288)\n    generations = raw_row.get(\"generations\")\n    correctness = raw_row.get(\"correctness_math_verify\")\n    completeness = raw_row.get(\"is_reasoning_complete\")\n    if not isinstance(generations, Sequence) or isinstance(generations, (str, bytes)):\n        raise SourceRowRejected(\"generations must be an array\")\n    if not isinstance(correctness, Sequence) or isinstance(correctness, (str, bytes)):\n        raise SourceRowRejected(\"correctness_math_verify must be an array\")\n    if not isinstance(completeness, Sequence) or isinstance(completeness, (str, bytes)):\n        raise SourceRowRejected(\"is_reasoning_complete must be an array\")\n    if not (len(generations) == len(correctness) == len(completeness)):\n        raise SourceRowRejected(\"math verification arrays do not have matching lengths\")\n    verified_reasoning = [\n        extract_reasoning_span(generation)\n        for generation, is_correct, is_complete in zip(generations, correctness, completeness, strict=True)\n        if isinstance(generation, str) and is_correct is True and is_complete is True\n    ]\n    if not verified_reasoning:\n        raise SourceRowRejected(\"row has no complete Math-Verify-correct reasoning generation\")\n    identifier = require_text(raw_row.get(\"uuid\", fallback_identifier), \"uuid\")\n    reasoning = min(verified_reasoning, key=len)\n    return bounded_record(\n        f\"openr1_math:{identifier}\",\n        problem,\n        answer,\n        {\n            \"source\": \"open-r1/OpenR1-Math-220k\",\n            \"revision\": SOURCE_REVISIONS[\"openr1_math\"],\n            \"verification\": \"math_verify_and_complete\",\n        },\n        MATH_SYSTEM_PROMPT,\n        reasoning,\n    )\n\n\ndef is_priority_code_record(record: DatasetRecord) -> bool:\n    candidate_text = f\"{record.input_text}\\n{record.output_text}\".lower()\n    return any(term in candidate_text for term in PRIORITY_CODE_TERMS)\n\n\ndef load_streaming_dataset(dataset_name: str, revision: str, config_name: str | None, seed: int, buffer_size: int):\n    try:\n        from datasets import load_dataset\n    except ModuleNotFoundError as error:\n        raise RuntimeError(\"install datasets==3.6.0 before running the A100 trainer\") from error\n    dataset = load_dataset(dataset_name, name=config_name, split=\"train\", streaming=True, revision=revision)\n    return dataset.shuffle(seed=seed, buffer_size=buffer_size)\n\n\ndef rejection_key(error: Exception) -> str:\n    message = str(error).split(\":\", maxsplit=1)[0]\n    return f\"{type(error).__name__}: {message}\"[:180]\n\n\ndef collect_source_records(\n    stream,\n    target_count: int,\n    scan_limit: int,\n    adapter: Callable[[Mapping[str, Any], str], DatasetRecord],\n    source_name: str,\n) -> tuple[list[DatasetRecord], dict[str, Any]]:\n    records: list[DatasetRecord] = []\n    rejections: Counter[str] = Counter()\n    scanned = 0\n    for raw_row in stream:\n        scanned += 1\n        if scanned > scan_limit:\n            break\n        if not isinstance(raw_row, Mapping):\n            rejections[\"SourceRowRejected: row is not an object\"] += 1\n            continue\n        try:\n            records.append(adapter(raw_row, str(scanned - 1)))\n        except (DatasetValidationError, SourceRowRejected, TypeError, ValueError, json.JSONDecodeError) as error:\n            rejections[rejection_key(error)] += 1\n        if len(records) == target_count:\n            return records, {\"source\": source_name, \"selected\": len(records), \"scanned\": scanned, \"rejections\": dict(rejections)}\n    raise RuntimeError(\n        f\"{source_name} produced {len(records)} valid records, below target {target_count}, after {scanned} scanned rows; rejections={dict(rejections)}\"\n    )\n\n\ndef collect_opencode_records(stream, quotas: CorpusQuotas, scan_limit: int) -> tuple[list[DatasetRecord], dict[str, Any]]:\n    priority_records: list[DatasetRecord] = []\n    general_records: list[DatasetRecord] = []\n    rejections: Counter[str] = Counter()\n    scanned = 0\n    for raw_row in stream:\n        scanned += 1\n        if scanned > scan_limit:\n            break\n        if not isinstance(raw_row, Mapping):\n            rejections[\"SourceRowRejected: row is not an object\"] += 1\n            continue\n        try:\n            record = adapt_opencode_row(raw_row, str(scanned - 1))\n        except (DatasetValidationError, SourceRowRejected, TypeError, ValueError, json.JSONDecodeError) as error:\n            rejections[rejection_key(error)] += 1\n            continue\n        if is_priority_code_record(record) and len(priority_records) < quotas.opencode_priority:\n            priority_records.append(record)\n        elif len(general_records) < quotas.opencode_general:\n            general_records.append(record)\n        else:\n            rejections[\"SourceRowRejected: quota bucket already full\"] += 1\n        if len(priority_records) == quotas.opencode_priority and len(general_records) == quotas.opencode_general:\n            return priority_records + general_records, {\n                \"source\": \"nvidia/OpenCodeInstruct\",\n                \"selected\": len(priority_records) + len(general_records),\n                \"priority_selected\": len(priority_records),\n                \"general_selected\": len(general_records),\n                \"scanned\": scanned,\n                \"rejections\": dict(rejections),\n            }\n    raise RuntimeError(\n        \"nvidia/OpenCodeInstruct produced \"\n        f\"priority={len(priority_records)}/{quotas.opencode_priority} and \"\n        f\"general={len(general_records)}/{quotas.opencode_general} after {scanned} scanned rows; \"\n        f\"rejections={dict(rejections)}\"\n    )\n\n\ndef record_to_json(record: DatasetRecord) -> dict[str, Any]:\n    return {\n        \"id\": record.identifier,\n        \"input\": record.input_text,\n        \"thinking\": record.thinking_text,\n        \"output\": record.output_text,\n        \"metadata\": record.metadata,\n        \"system\": record.system_text,\n    }\n\n\ndef record_from_json(value: Mapping[str, Any]) -> DatasetRecord:\n    identifier = require_text(value.get(\"id\"), \"id\")\n    input_text = require_text(value.get(\"input\"), \"input\")\n    output_text = require_text(value.get(\"output\"), \"output\")\n    thinking_text = value.get(\"thinking\")\n    system_text = value.get(\"system\")\n    metadata = value.get(\"metadata\")\n    if thinking_text is not None and not isinstance(thinking_text, str):\n        raise DatasetValidationError(\"field 'thinking' must be a string or null\")\n    if system_text is not None and not isinstance(system_text, str):\n        raise DatasetValidationError(\"field 'system' must be a string or null\")\n    if not isinstance(metadata, dict):\n        raise DatasetValidationError(\"field 'metadata' must be an object\")\n    return DatasetRecord(identifier, input_text, thinking_text, output_text, metadata, system_text)\n\n\ndef write_corpus(path: Path, records: Sequence[DatasetRecord]) -> None:\n    temporary_path = path.with_name(f\".{path.name}.{uuid.uuid4().hex}.tmp\")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    try:\n        with temporary_path.open(\"w\", encoding=\"utf-8\", newline=\"\\n\") as corpus_file:\n            for record in records:\n                corpus_file.write(json.dumps(record_to_json(record), ensure_ascii=False, sort_keys=True))\n                corpus_file.write(\"\\n\")\n        os.replace(temporary_path, path)\n    except BaseException:\n        if temporary_path.exists():\n            temporary_path.unlink()\n        raise\n\n\ndef read_corpus(path: Path) -> list[DatasetRecord]:\n    records: list[DatasetRecord] = []\n    with path.open(\"r\", encoding=\"utf-8\") as corpus_file:\n        for line_number, line in enumerate(corpus_file, start=1):\n            if not line.strip():\n                raise DatasetValidationError(f\"corpus line {line_number} is empty\")\n            value = json.loads(line)\n            if not isinstance(value, Mapping):\n                raise DatasetValidationError(f\"corpus line {line_number} is not an object\")\n            records.append(record_from_json(value))\n    if not records:\n        raise DatasetValidationError(\"corpus is empty\")\n    identifiers = [record.identifier for record in records]\n    if len(identifiers) != len(set(identifiers)):\n        raise DatasetValidationError(\"corpus contains duplicate record identifiers\")\n    return records\n\n\ndef build_or_load_corpus(configuration: RunConfiguration) -> tuple[list[DatasetRecord], dict[str, Any]]:\n    corpus_path = configuration.results_directory / \"corpus.jsonl\"\n    manifest_path = configuration.results_directory / \"corpus_manifest.json\"\n    corpus_exists = corpus_path.exists()\n    manifest_exists = manifest_path.exists()\n    requested_contract = {\n        \"corpus_format_version\": CORPUS_FORMAT_VERSION,\n        \"source_revisions\": SOURCE_REVISIONS,\n        \"quotas\": configuration.quotas.to_dict(),\n        \"data_seed\": configuration.data_seed,\n        \"shuffle_buffer_size\": configuration.shuffle_buffer_size,\n    }\n    if corpus_exists and manifest_exists:\n        manifest = read_json_object(manifest_path)\n        if manifest.get(\"requested_contract\") != requested_contract:\n            raise ValueError(\"existing corpus contract does not match this A100 run\")\n        actual_hash = sha256_file(corpus_path)\n        if manifest.get(\"sha256\") != actual_hash:\n            raise ValueError(\"existing corpus SHA-256 does not match its manifest\")\n        records = read_corpus(corpus_path)\n        if manifest.get(\"record_count\") != len(records):\n            raise ValueError(\"existing corpus record count does not match its manifest\")\n        return records, manifest\n    recovered_incomplete_artifact = corpus_exists != manifest_exists\n\n    opencode_stream = load_streaming_dataset(\n        \"nvidia/OpenCodeInstruct\",\n        SOURCE_REVISIONS[\"opencode\"],\n        \"train\",\n        configuration.data_seed,\n        configuration.shuffle_buffer_size,\n    )\n    opencode_records, opencode_report = collect_opencode_records(\n        opencode_stream,\n        configuration.quotas,\n        configuration.opencode_scan_limit,\n    )\n    codefeedback_stream = load_streaming_dataset(\n        \"m-a-p/CodeFeedback-Filtered-Instruction\",\n        SOURCE_REVISIONS[\"codefeedback\"],\n        None,\n        configuration.data_seed + 1,\n        configuration.shuffle_buffer_size,\n    )\n    codefeedback_records, codefeedback_report = collect_source_records(\n        codefeedback_stream,\n        configuration.quotas.codefeedback,\n        configuration.source_scan_limit,\n        adapt_codefeedback_row,\n        \"m-a-p/CodeFeedback-Filtered-Instruction\",\n    )\n    magicoder_stream = load_streaming_dataset(\n        \"ise-uiuc/Magicoder-Evol-Instruct-110K\",\n        SOURCE_REVISIONS[\"magicoder\"],\n        None,\n        configuration.data_seed + 2,\n        configuration.shuffle_buffer_size,\n    )\n    magicoder_records, magicoder_report = collect_source_records(\n        magicoder_stream,\n        configuration.quotas.magicoder,\n        configuration.source_scan_limit,\n        adapt_magicoder_row,\n        \"ise-uiuc/Magicoder-Evol-Instruct-110K\",\n    )\n    math_stream = load_streaming_dataset(\n        \"open-r1/OpenR1-Math-220k\",\n        SOURCE_REVISIONS[\"openr1_math\"],\n        \"default\",\n        configuration.data_seed + 3,\n        configuration.shuffle_buffer_size,\n    )\n    math_records, math_report = collect_source_records(\n        math_stream,\n        configuration.quotas.openr1_math,\n        configuration.source_scan_limit,\n        adapt_openr1_math_row,\n        \"open-r1/OpenR1-Math-220k\",\n    )\n    records = opencode_records + codefeedback_records + magicoder_records + math_records\n    identifiers = [record.identifier for record in records]\n    if len(identifiers) != len(set(identifiers)):\n        raise DatasetValidationError(\"selected corpus contains duplicate record identifiers\")\n    write_corpus(corpus_path, records)\n    manifest = {\n        \"requested_contract\": requested_contract,\n        \"record_count\": len(records),\n        \"sha256\": sha256_file(corpus_path),\n        \"sources\": [opencode_report, codefeedback_report, magicoder_report, math_report],\n        \"recovered_incomplete_artifact\": recovered_incomplete_artifact,\n    }\n    atomic_write_json(manifest_path, manifest)\n    return records, manifest\n\n\ndef split_records(records: Sequence[DatasetRecord], data_seed: int, validation_fraction: float = 0.03) -> tuple[list[DatasetRecord], list[DatasetRecord]]:\n    if not 0.0 < validation_fraction < 1.0:\n        raise ValueError(\"validation_fraction must be between zero and one\")\n    training_records: list[DatasetRecord] = []\n    validation_records: list[DatasetRecord] = []\n    threshold = int(validation_fraction * 10_000)\n    for record in records:\n        digest = hashlib.sha256(f\"{data_seed}:{record.identifier}\".encode(\"utf-8\")).digest()\n        bucket = int.from_bytes(digest[:4], \"big\") % 10_000\n        (validation_records if bucket < threshold else training_records).append(record)\n    if not training_records or not validation_records:\n        raise DatasetValidationError(\"deterministic split produced an empty partition\")\n    return training_records, validation_records\n\n\ndef model_settings() -> ModelSettings:\n    return ModelSettings(\n        embedding_size=512,\n        memory_features=16,\n        local_memory_size=16,\n        salience_memory_size=16,\n        salience_threshold=0.75,\n        expert_count=128,\n        expert_top_k=6,\n        cache_capacity=256,\n        scan_chunk=128,\n        refine_decay_rate=0.0625,\n        ablation=\"no_refine\",\n    )\n\n\ndef configure_a100() -> tuple[torch.device, dict[str, Any]]:\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"this notebook requires a CUDA A100 runtime\")\n    device = torch.device(\"cuda\")\n    properties = torch.cuda.get_device_properties(device)\n    gpu_name = torch.cuda.get_device_name(device)\n    if \"A100\" not in gpu_name.upper():\n        raise RuntimeError(f\"this run requires an A100, received '{gpu_name}'\")\n    if properties.total_memory < MINIMUM_A100_MEMORY_BYTES:\n        raise RuntimeError(f\"this run requires at least 70 GiB VRAM, received {properties.total_memory / 2**30:.2f} GiB\")\n    if properties.major < 8:\n        raise RuntimeError(f\"this run requires compute capability >= 8.0, received {properties.major}.{properties.minor}\")\n    if not torch.cuda.is_bf16_supported():\n        raise RuntimeError(\"this A100 runtime does not expose CUDA BF16 support\")\n    torch.set_float32_matmul_precision(\"high\")\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    return device, {\n        \"gpu\": gpu_name,\n        \"total_memory_bytes\": properties.total_memory,\n        \"compute_capability\": [properties.major, properties.minor],\n        \"torch\": torch.__version__,\n        \"torch_cuda\": torch.version.cuda,\n        \"bf16\": True,\n        \"tf32_matmul\": torch.backends.cuda.matmul.allow_tf32,\n        \"tf32_cudnn\": torch.backends.cudnn.allow_tf32,\n    }\n\n\ndef state_max_abs_difference(left_state, right_state) -> dict[str, float]:\n    fields = (\n        \"working_state\",\n        \"memory_basis\",\n        \"memory_normalizer\",\n        \"refine_basis\",\n        \"refine_normalizer\",\n        \"local_keys\",\n        \"local_values\",\n        \"local_valid\",\n        \"salient_keys\",\n        \"salient_values\",\n        \"salient_valid\",\n        \"last_token_ids\",\n    )\n    differences: dict[str, float] = {}\n    for field_name in fields:\n        left_value = getattr(left_state, field_name)\n        right_value = getattr(right_state, field_name)\n        if left_value.dtype in {torch.bool, torch.long}:\n            differences[field_name] = 0.0 if torch.equal(left_value, right_value) else float(\"inf\")\n        else:\n            differences[field_name] = float((left_value - right_value).abs().max().cpu())\n    differences[\"step_index\"] = 0.0 if left_state.step_index == right_state.step_index else float(\"inf\")\n    return differences\n\n\ndef run_cuda_preflight(model: KoemiModel, device: torch.device) -> dict[str, Any]:\n    model.eval()\n    contract_input = torch.randint(0, 256, (2, 32), device=device)\n    with torch.inference_mode():\n        parallel = model(contract_input, execution_mode=ExecutionMode.PARALLEL)\n        sequential = model(contract_input, execution_mode=ExecutionMode.SEQUENTIAL)\n    logits_error = float((parallel.logits - sequential.logits).abs().max().cpu())\n    state_error = state_max_abs_difference(parallel.state, sequential.state)\n    if logits_error > 1e-3 or any(not math.isfinite(value) or value > 1e-3 for value in state_error.values()):\n        raise AssertionError(f\"parallel/sequential HERM contract failed: logits={logits_error}, state={state_error}\")\n    if not torch.equal(parallel.active_expert_indices, sequential.active_expert_indices):\n        raise AssertionError(\"parallel/sequential expert assignments differ\")\n    numerator = torch.ones(1, model.settings.embedding_size, device=device)\n    denominator = torch.zeros(1, 1, device=device)\n    empty_read = model.associative_memory.confidence_weighted_read(numerator, denominator)\n    if not torch.equal(empty_read, torch.zeros_like(empty_read)):\n        raise AssertionError(\"zero-evidence associative read is not zero\")\n    model.train()\n    return {\n        \"parallel_sequential_logits_max_abs\": logits_error,\n        \"parallel_sequential_state_max_abs\": state_error,\n        \"zero_evidence_read_max_abs\": float(empty_read.abs().max().cpu()),\n        \"expert_assignments_equal\": True,\n    }\n\n\ndef move_batch(batch: dict[str, Tensor], device: torch.device) -> tuple[Tensor, Tensor, Tensor]:\n    return (\n        batch[\"input_ids\"].to(device, non_blocking=True),\n        batch[\"target_ids\"].to(device, non_blocking=True),\n        batch[\"thinking_mask\"].to(device, non_blocking=True),\n    )\n\n\ndef make_probe_batch(dataset: MaterializedCausalByteDataset, batch_size: int) -> dict[str, Tensor]:\n    chunk_count = min(len(dataset), batch_size)\n    chunks = [dataset[index % chunk_count] for index in range(batch_size)]\n    return collate_materialized_chunks(chunks)\n\n\ndef benchmark_batch_size(\n    settings: ModelSettings,\n    dataset: MaterializedCausalByteDataset,\n    device: torch.device,\n    model_seed: int,\n    candidate_batch_size: int,\n) -> dict[str, Any]:\n    torch.manual_seed(model_seed)\n    torch.cuda.manual_seed_all(model_seed)\n    candidate_model = KoemiModel(settings).to(device)\n    optimizer = torch.optim.AdamW(candidate_model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=False)\n    batch = make_probe_batch(dataset, candidate_batch_size)\n    input_ids, target_ids, thinking_mask = move_batch(batch, device)\n    try:\n        for _ in range(2):\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16):\n                output = candidate_model(input_ids, execution_mode=ExecutionMode.PARALLEL)\n                objective = calculate_training_objective(output, target_ids, thinking_mask, 0.5)\n            scaler.scale(objective.total_loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n        torch.cuda.synchronize(device)\n        torch.cuda.reset_peak_memory_stats(device)\n        started_at = time.perf_counter()\n        supervised_tokens = 0\n        for _ in range(3):\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16):\n                output = candidate_model(input_ids, execution_mode=ExecutionMode.PARALLEL)\n                objective = calculate_training_objective(output, target_ids, thinking_mask, 0.5)\n            if not bool(torch.isfinite(objective.total_loss)):\n                raise FloatingPointError(\"non-finite objective in A100 batch calibration\")\n            scaler.scale(objective.total_loss).backward()\n            gradient_norm = torch.nn.utils.clip_grad_norm_(candidate_model.parameters(), 1.0)\n            if not bool(torch.isfinite(gradient_norm)):\n                raise FloatingPointError(\"non-finite gradient in A100 batch calibration\")\n            scaler.step(optimizer)\n            scaler.update()\n            supervised_tokens += int((target_ids != IGNORE_TARGET_ID).sum().item())\n        torch.cuda.synchronize(device)\n        elapsed_seconds = time.perf_counter() - started_at\n        return {\n            \"batch_size\": candidate_batch_size,\n            \"status\": \"ok\",\n            \"supervised_tokens_per_second\": supervised_tokens / elapsed_seconds,\n            \"peak_memory_bytes\": torch.cuda.max_memory_allocated(device),\n            \"elapsed_seconds\": elapsed_seconds,\n        }\n    except torch.cuda.OutOfMemoryError as error:\n        return {\"batch_size\": candidate_batch_size, \"status\": \"oom\", \"error\": str(error).splitlines()[0]}\n    finally:\n        del candidate_model, optimizer, scaler, batch, input_ids, target_ids, thinking_mask\n        torch.cuda.empty_cache()\n\n\ndef calibrate_batch_size(\n    settings: ModelSettings,\n    dataset: MaterializedCausalByteDataset,\n    device: torch.device,\n    model_seed: int,\n) -> tuple[int, list[dict[str, Any]]]:\n    candidates = (4, 8, 12, 16, 24, 32)\n    reports = [benchmark_batch_size(settings, dataset, device, model_seed, candidate) for candidate in candidates]\n    safe_reports = [\n        report\n        for report in reports\n        if report[\"status\"] == \"ok\" and report[\"peak_memory_bytes\"] <= int(MINIMUM_A100_MEMORY_BYTES * 0.85)\n    ]\n    if not safe_reports:\n        raise RuntimeError(f\"no calibrated batch fits the A100 safety budget: {reports}\")\n    selected = max(safe_reports, key=lambda report: report[\"supervised_tokens_per_second\"])\n    return int(selected[\"batch_size\"]), reports\n\n\ndef create_loader(\n    dataset: MaterializedCausalByteDataset,\n    batch_size: int,\n    data_seed: int,\n    epoch_index: int,\n    start_batch_index: int,\n    num_workers: int,\n) -> DataLoader[dict[str, Tensor]]:\n    sampler = DeterministicBatchSampler(len(dataset), batch_size, data_seed, epoch_index, start_batch_index)\n    options: dict[str, Any] = {}\n    if num_workers > 0:\n        options[\"prefetch_factor\"] = 2\n    return DataLoader(\n        dataset,\n        batch_sampler=sampler,\n        collate_fn=collate_materialized_chunks,\n        num_workers=num_workers,\n        pin_memory=True,\n        **options,\n    )\n\n\ndef schedule_lambda(step: int, warmup_steps: int, total_steps: int) -> float:\n    if step < warmup_steps:\n        return (step + 1) / max(1, warmup_steps)\n    progress = min(1.0, max(0.0, (step - warmup_steps) / max(1, total_steps - warmup_steps)))\n    return 0.5 * (1.0 + math.cos(math.pi * progress))\n\n\ndef training_payload(\n    model: KoemiModel,\n    optimizer: torch.optim.Optimizer,\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\n    scaler: torch.amp.GradScaler,\n    training_state: Mapping[str, int],\n) -> dict[str, Any]:\n    return {\n        \"model_state\": model.state_dict(),\n        \"optimizer_state\": optimizer.state_dict(),\n        \"scheduler_state\": scheduler.state_dict(),\n        \"scaler_state\": scaler.state_dict(),\n        \"training_state\": dict(training_state),\n        \"python_random_state\": random.getstate(),\n        \"torch_random_state\": torch.get_rng_state(),\n        \"cuda_random_states\": torch.cuda.get_rng_state_all(),\n    }\n\n\ndef restore_training_payload(\n    payload: Mapping[str, Any],\n    model: KoemiModel,\n    optimizer: torch.optim.Optimizer,\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\n    scaler: torch.amp.GradScaler,\n) -> dict[str, int]:\n    required = (\"model_state\", \"optimizer_state\", \"scheduler_state\", \"scaler_state\", \"training_state\")\n    missing = [field for field in required if field not in payload]\n    if missing:\n        raise ValueError(f\"checkpoint is missing fields: {missing}\")\n    training_state = payload[\"training_state\"]\n    if not isinstance(training_state, Mapping):\n        raise ValueError(\"checkpoint training_state is invalid\")\n    restored_state = {name: int(training_state[name]) for name in (\"epoch_index\", \"next_batch_index\", \"tokens_seen\")}\n    model.load_state_dict(payload[\"model_state\"])\n    optimizer.load_state_dict(payload[\"optimizer_state\"])\n    scheduler.load_state_dict(payload[\"scheduler_state\"])\n    scaler.load_state_dict(payload[\"scaler_state\"])\n    if \"python_random_state\" in payload:\n        random.setstate(payload[\"python_random_state\"])\n    if \"torch_random_state\" in payload:\n        torch.set_rng_state(payload[\"torch_random_state\"])\n    if \"cuda_random_states\" in payload:\n        torch.cuda.set_rng_state_all(payload[\"cuda_random_states\"])\n    return restored_state\n\n\ndef append_json_line(path: Path, value: Mapping[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"a\", encoding=\"utf-8\", newline=\"\\n\") as log_file:\n        log_file.write(json.dumps(value, sort_keys=True))\n        log_file.write(\"\\n\")\n\n\ndef evaluate(\n    model: KoemiModel,\n    dataset: MaterializedCausalByteDataset,\n    batch_size: int,\n    device: torch.device,\n    maximum_batches: int,\n    num_workers: int,\n) -> dict[str, Any]:\n    options: dict[str, Any] = {}\n    if num_workers > 0:\n        options[\"prefetch_factor\"] = 2\n    loader = DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        collate_fn=collate_materialized_chunks,\n        num_workers=num_workers,\n        pin_memory=True,\n        **options,\n    )\n    metrics = RunningMetrics(device, model.settings.expert_count)\n    model.eval()\n    torch.cuda.reset_peak_memory_stats(device)\n    started_at = time.perf_counter()\n    with torch.inference_mode():\n        for batch_index, batch in enumerate(loader):\n            if batch_index >= maximum_batches:\n                break\n            input_ids, target_ids, thinking_mask = move_batch(batch, device)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16):\n                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)\n                token_losses = token_cross_entropy(output.logits.float(), target_ids)\n            if not bool(torch.isfinite(token_losses[target_ids != IGNORE_TARGET_ID]).all()):\n                raise FloatingPointError(\"non-finite validation token loss\")\n            metrics.add(output, token_losses, target_ids, thinking_mask)\n    torch.cuda.synchronize(device)\n    elapsed_seconds = time.perf_counter() - started_at\n    report = metrics.as_dict(elapsed_seconds)\n    report[\"batches\"] = min(maximum_batches, math.ceil(len(dataset) / batch_size))\n    report[\"peak_memory_bytes\"] = torch.cuda.max_memory_allocated(device)\n    model.train()\n    return report\n\n\ndef expert_load_summary(counts: Sequence[int]) -> dict[str, Any]:\n    total = sum(counts)\n    if not counts:\n        return {\"counts\": [], \"total_assignments\": 0, \"occupied_experts\": 0, \"entropy_nats\": 0.0, \"normalized_entropy\": 0.0, \"load_gini\": 0.0}\n    probabilities = [count / total for count in counts if count > 0] if total else []\n    entropy = -sum(probability * math.log(probability) for probability in probabilities)\n    mean = total / len(counts) if counts else 0.0\n    absolute_difference_total = sum(abs(left - right) for left in counts for right in counts)\n    gini = absolute_difference_total / (2 * len(counts) * total) if total else 0.0\n    return {\n        \"counts\": list(counts),\n        \"total_assignments\": total,\n        \"occupied_experts\": sum(count > 0 for count in counts),\n        \"entropy_nats\": entropy,\n        \"normalized_entropy\": entropy / math.log(len(counts)) if len(counts) > 1 else 0.0,\n        \"load_min\": min(counts),\n        \"load_max\": max(counts),\n        \"load_mean\": mean,\n        \"load_gini\": gini,\n    }\n\n\ndef run_training(\n    configuration: RunConfiguration,\n    corpus_manifest: Mapping[str, Any],\n    training_dataset: MaterializedCausalByteDataset,\n    validation_dataset: MaterializedCausalByteDataset,\n    device: torch.device,\n    environment: Mapping[str, Any],\n) -> dict[str, Any]:\n    settings = model_settings()\n    run_manifest_path = configuration.results_directory / \"run_manifest.json\"\n    checkpoint_root = configuration.results_directory / \"checkpoints\"\n    if run_manifest_path.exists():\n        run_manifest = read_json_object(run_manifest_path)\n        if run_manifest.get(\"corpus_sha256\") != corpus_manifest.get(\"sha256\"):\n            raise ValueError(\"run manifest corpus hash does not match the selected corpus\")\n        if run_manifest.get(\"model_settings\") != settings.to_dict():\n            raise ValueError(\"run manifest model settings do not match\")\n        selected_batch_size = run_manifest.get(\"selected_batch_size\")\n        if not isinstance(selected_batch_size, int) or selected_batch_size < 1:\n            raise ValueError(\"run manifest selected batch size is invalid\")\n        calibration = run_manifest.get(\"batch_calibration\")\n        if not isinstance(calibration, list):\n            raise ValueError(\"run manifest batch calibration is invalid\")\n    else:\n        selected_batch_size, calibration = calibrate_batch_size(settings, training_dataset, device, configuration.model_seed)\n        run_manifest = {\n            \"run_format_version\": RUN_FORMAT_VERSION,\n            \"corpus_sha256\": corpus_manifest[\"sha256\"],\n            \"model_settings\": settings.to_dict(),\n            \"selected_batch_size\": selected_batch_size,\n            \"target_effective_batch_size\": 64,\n            \"batch_calibration\": calibration,\n            \"environment\": dict(environment),\n            \"source_revisions\": SOURCE_REVISIONS,\n            \"sequence_length\": configuration.sequence_length,\n            \"thinking_loss_weight\": 0.5,\n        }\n        atomic_write_json(run_manifest_path, run_manifest)\n    gradient_accumulation_steps = math.ceil(run_manifest[\"target_effective_batch_size\"] / selected_batch_size)\n    run_contract = {\n        \"run_format_version\": RUN_FORMAT_VERSION,\n        \"corpus_sha256\": corpus_manifest[\"sha256\"],\n        \"model_settings\": settings.to_dict(),\n        \"selected_batch_size\": selected_batch_size,\n        \"gradient_accumulation_steps\": gradient_accumulation_steps,\n        \"thinking_loss_weight\": 0.5,\n        \"learning_rate\": 3e-4,\n        \"weight_decay\": 0.01,\n        \"warmup_steps\": 1_000,\n        \"schedule_steps\": 100_000,\n        \"precision\": \"bf16\",\n    }\n    run_signature = fingerprint(run_contract)\n    checkpoint_store = RotatingCheckpointStore(checkpoint_root, run_signature)\n    torch.manual_seed(configuration.model_seed)\n    torch.cuda.manual_seed_all(configuration.model_seed)\n    model = KoemiModel(settings).to(device)\n    preflight_report = run_cuda_preflight(model, device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: schedule_lambda(step, 1_000, 100_000))\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=False)\n    checkpoint_result = checkpoint_store.load_latest()\n    training_state = {\"epoch_index\": 0, \"next_batch_index\": 0, \"tokens_seen\": 0}\n    if checkpoint_result.payload is not None:\n        training_state = restore_training_payload(checkpoint_result.payload, model, optimizer, scheduler, scaler)\n    step_log_path = configuration.results_directory / \"training_steps.jsonl\"\n    started_at = time.perf_counter()\n    last_checkpoint_at = started_at\n    optimizer_step = int(checkpoint_result.payload[\"optimizer_step\"]) if checkpoint_result.payload is not None else 0\n    session_step_count = 0\n    stopped_for_budget = False\n    while time.perf_counter() - started_at < configuration.session_seconds:\n        loader = create_loader(\n            training_dataset,\n            selected_batch_size,\n            configuration.data_seed,\n            training_state[\"epoch_index\"],\n            training_state[\"next_batch_index\"],\n            configuration.num_workers,\n        )\n        if len(loader) == 0:\n            training_state[\"epoch_index\"] += 1\n            training_state[\"next_batch_index\"] = 0\n            continue\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        metrics = RunningMetrics(device, settings.expert_count)\n        optimizer_window_started_at = time.perf_counter()\n        accumulated_batches = 0\n        starting_batch_index = training_state[\"next_batch_index\"]\n        last_consumed_batch = training_state[\"next_batch_index\"]\n        for relative_batch_index, batch in enumerate(loader):\n            if time.perf_counter() - started_at >= configuration.session_seconds:\n                stopped_for_budget = True\n                break\n            absolute_batch_index = starting_batch_index + relative_batch_index\n            input_ids, target_ids, thinking_mask = move_batch(batch, device)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16):\n                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)\n                objective = calculate_training_objective(output, target_ids, thinking_mask, 0.5)\n                scaled_loss = objective.total_loss / gradient_accumulation_steps\n            if not bool(torch.isfinite(objective.total_loss)):\n                raise FloatingPointError(f\"non-finite training objective at optimizer step {optimizer_step + 1}\")\n            scaler.scale(scaled_loss).backward()\n            token_losses = token_cross_entropy(output.logits.float(), target_ids)\n            metrics.add(output, token_losses, target_ids, thinking_mask)\n            accumulated_batches += 1\n            last_consumed_batch = absolute_batch_index + 1\n            if accumulated_batches < gradient_accumulation_steps:\n                continue\n            scaler.unscale_(optimizer)\n            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not bool(torch.isfinite(gradient_norm)):\n                raise FloatingPointError(f\"non-finite gradient at optimizer step {optimizer_step + 1}\")\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            optimizer.zero_grad(set_to_none=True)\n            torch.cuda.synchronize(device)\n            optimizer_step += 1\n            session_step_count += 1\n            training_state[\"next_batch_index\"] = last_consumed_batch\n            step_elapsed_seconds = time.perf_counter() - optimizer_window_started_at\n            step_report = metrics.as_dict(step_elapsed_seconds)\n            step_report.update(\n                {\n                    \"optimizer_step\": optimizer_step,\n                    \"epoch_index\": training_state[\"epoch_index\"],\n                    \"next_batch_index\": training_state[\"next_batch_index\"],\n                    \"tokens_seen\": training_state[\"tokens_seen\"] + step_report[\"supervised_tokens\"],\n                    \"gradient_norm\": float(gradient_norm.cpu()),\n                    \"learning_rate\": optimizer.param_groups[0][\"lr\"],\n                    \"gpu_allocated_bytes\": torch.cuda.memory_allocated(device),\n                    \"gpu_peak_allocated_bytes\": torch.cuda.max_memory_allocated(device),\n                }\n            )\n            training_state[\"tokens_seen\"] = int(step_report[\"tokens_seen\"])\n            if optimizer_step % configuration.log_interval_steps == 0:\n                append_json_line(step_log_path, step_report)\n                print(json.dumps(step_report, sort_keys=True))\n            now = time.perf_counter()\n            if now - last_checkpoint_at >= configuration.checkpoint_interval_seconds:\n                checkpoint_store.save(training_payload(model, optimizer, scheduler, scaler, training_state), optimizer_step)\n                last_checkpoint_at = now\n            metrics = RunningMetrics(device, settings.expert_count)\n            optimizer_window_started_at = time.perf_counter()\n            accumulated_batches = 0\n        if stopped_for_budget:\n            optimizer.zero_grad(set_to_none=True)\n            break\n        if accumulated_batches:\n            optimizer.zero_grad(set_to_none=True)\n        training_state[\"epoch_index\"] += 1\n        training_state[\"next_batch_index\"] = 0\n    final_checkpoint_path = checkpoint_store.save(training_payload(model, optimizer, scheduler, scaler, training_state), optimizer_step)\n    validation_report = evaluate(\n        model,\n        validation_dataset,\n        selected_batch_size,\n        device,\n        configuration.evaluation_batches,\n        configuration.num_workers,\n    )\n    model_checkpoint_path = configuration.results_directory / \"koemi-3hip-a100-code-reasoning-model.pt\"\n    CheckpointStore().save(model_checkpoint_path, model, overwrite=True)\n    generated_text = generate_text(\n        model,\n        ByteTokenizer(),\n        \"Diagnose this Python traceback and state a testable fix:\\n\",\n        max_new_bytes=256,\n        temperature=0.7,\n        device=device,\n    )\n    sample_path = configuration.results_directory / \"sample_generation.txt\"\n    atomic_write_text(sample_path, generated_text)\n    session_report = {\n        \"run_signature\": run_signature,\n        \"run_contract\": run_contract,\n        \"preflight\": preflight_report,\n        \"checkpoint_recovery\": list(checkpoint_result.recovery_messages),\n        \"training_state\": training_state,\n        \"optimizer_step\": optimizer_step,\n        \"session_optimizer_steps\": session_step_count,\n        \"session_wall_seconds\": time.perf_counter() - started_at,\n        \"stopped_for_budget\": stopped_for_budget,\n        \"checkpoint\": str(final_checkpoint_path),\n        \"model_checkpoint\": str(model_checkpoint_path),\n        \"sample_generation\": str(sample_path),\n        \"validation\": validation_report,\n    }\n    atomic_write_json(configuration.results_directory / \"latest_session.json\", session_report)\n    return session_report\n\n\ndef run_internal_contract_tests() -> dict[str, Any]:\n    opencode = adapt_opencode_row(\n        {\n            \"id\": \"ok\",\n            \"input\": \"Fix this Python traceback.\",\n            \"output\": \"Check the exception and add a test.\",\n            \"domain\": \"debugging\",\n            \"average_test_score\": \"1.0\",\n            \"tests_execution_status\": '[\"pass\", \"pass\"]',\n        },\n        \"0\",\n    )\n    if not is_priority_code_record(opencode):\n        raise AssertionError(\"priority code classification rejected a Python traceback\")\n    math_record = adapt_openr1_math_row(\n        {\n            \"uuid\": \"math-ok\",\n            \"problem\": \"What is 1 + 1?\",\n            \"answer\": \"2\",\n            \"generations\": [\"<think>Add one and one to get two.</think>\\\\n2\"],\n            \"correctness_math_verify\": [True],\n            \"is_reasoning_complete\": [True],\n        },\n        \"0\",\n    )\n    reference_dataset = CausalByteDataset((opencode, math_record), sequence_length=16)\n    materialized_dataset = MaterializedCausalByteDataset((opencode, math_record), sequence_length=16)\n    reference_chunks = [\n        chunk\n        for chunk in reference_dataset.chunks\n        if any(target != IGNORE_TARGET_ID for target in chunk.target_ids)\n    ]\n    if len(reference_chunks) != len(materialized_dataset):\n        raise AssertionError(\"materialized dataset chunk count differs from repository dataset\")\n    for index, reference_chunk in enumerate(reference_chunks):\n        input_bytes, target_bytes, supervised_bytes, thinking_bytes = materialized_dataset[index]\n        expected_targets = tuple(\n            target if supervised else IGNORE_TARGET_ID\n            for target, supervised in zip(target_bytes, supervised_bytes, strict=True)\n        )\n        if reference_chunk.input_ids != tuple(input_bytes) or reference_chunk.target_ids != expected_targets:\n            raise AssertionError(\"materialized dataset does not match repository causal chunks\")\n        if reference_chunk.thinking_mask != tuple(bool(value) for value in thinking_bytes):\n            raise AssertionError(\"materialized thinking mask does not match repository causal chunks\")\n    sampler = DeterministicBatchSampler(11, 3, 7, 2, 1)\n    full_sampler = DeterministicBatchSampler(11, 3, 7, 2, 0)\n    if list(sampler) != list(full_sampler)[1:]:\n        raise AssertionError(\"resumed deterministic sampler does not match the epoch suffix\")\n    with tempfile.TemporaryDirectory() as temporary_directory:\n        store = RotatingCheckpointStore(Path(temporary_directory), \"contract-test\")\n        store.save({\"training_state\": {}}, 1)\n        store.save({\"training_state\": {}}, 2)\n        newer_slot = store.slot_paths[2 % 2]\n        newer_slot.write_bytes(b\"corrupt\")\n        recovered = store.load_latest()\n        if recovered.payload is None or recovered.payload[\"optimizer_step\"] != 1:\n            raise AssertionError(\"rotating checkpoint store did not recover the older valid slot\")\n    return {\"status\": \"passed\", \"dataset_chunks\": len(materialized_dataset), \"checkpoint_recovery\": \"passed\"}\n\n\ndef parse_arguments() -> RunConfiguration:\n    parser = argparse.ArgumentParser(description=\"Run the Koemi-3HIP A100 code and reasoning experiment\")\n    parser.add_argument(\"--results-dir\", required=True)\n    parser.add_argument(\"--session-hours\", type=float, default=9.0)\n    parser.add_argument(\"--data-seed\", type=int, default=20260913)\n    parser.add_argument(\"--model-seed\", type=int, default=1337)\n    parser.add_argument(\"--sequence-length\", type=int, default=256)\n    parser.add_argument(\"--opencode-priority-records\", type=int, default=60_000)\n    parser.add_argument(\"--opencode-general-records\", type=int, default=120_000)\n    parser.add_argument(\"--codefeedback-records\", type=int, default=80_000)\n    parser.add_argument(\"--magicoder-records\", type=int, default=70_000)\n    parser.add_argument(\"--math-records\", type=int, default=35_000)\n    parser.add_argument(\"--opencode-scan-limit\", type=int, default=2_000_000)\n    parser.add_argument(\"--source-scan-limit\", type=int, default=200_000)\n    parser.add_argument(\"--shuffle-buffer-size\", type=int, default=20_000)\n    parser.add_argument(\"--num-workers\", type=int, default=2)\n    parser.add_argument(\"--checkpoint-minutes\", type=int, default=15)\n    parser.add_argument(\"--log-interval-steps\", type=int, default=10)\n    parser.add_argument(\"--evaluation-batches\", type=int, default=128)\n    arguments = parser.parse_args()\n    if arguments.session_hours <= 0:\n        raise ValueError(\"session-hours must be positive\")\n    if arguments.num_workers < 0:\n        raise ValueError(\"num-workers must be non-negative\")\n    positive_values = (\n        arguments.sequence_length,\n        arguments.opencode_priority_records,\n        arguments.opencode_general_records,\n        arguments.codefeedback_records,\n        arguments.magicoder_records,\n        arguments.math_records,\n        arguments.opencode_scan_limit,\n        arguments.source_scan_limit,\n        arguments.shuffle_buffer_size,\n        arguments.checkpoint_minutes,\n        arguments.log_interval_steps,\n        arguments.evaluation_batches,\n    )\n    if any(value < 1 for value in positive_values):\n        raise ValueError(\"all counts and intervals must be positive\")\n    return RunConfiguration(\n        results_directory=Path(arguments.results_dir).expanduser().resolve(),\n        session_seconds=round(arguments.session_hours * 60 * 60),\n        data_seed=arguments.data_seed,\n        model_seed=arguments.model_seed,\n        sequence_length=arguments.sequence_length,\n        quotas=CorpusQuotas(\n            arguments.opencode_priority_records,\n            arguments.opencode_general_records,\n            arguments.codefeedback_records,\n            arguments.magicoder_records,\n            arguments.math_records,\n        ),\n        opencode_scan_limit=arguments.opencode_scan_limit,\n        source_scan_limit=arguments.source_scan_limit,\n        shuffle_buffer_size=arguments.shuffle_buffer_size,\n        num_workers=arguments.num_workers,\n        checkpoint_interval_seconds=arguments.checkpoint_minutes * 60,\n        log_interval_steps=arguments.log_interval_steps,\n        evaluation_batches=arguments.evaluation_batches,\n    )\n\n\ndef main() -> None:\n    configuration = parse_arguments()\n    internal_tests = run_internal_contract_tests()\n    device, environment = configure_a100()\n    configuration.results_directory.mkdir(parents=True, exist_ok=True)\n    atomic_write_json(configuration.results_directory / \"environment.json\", environment)\n    atomic_write_json(configuration.results_directory / \"internal_contract_tests.json\", internal_tests)\n    records, corpus_manifest = build_or_load_corpus(configuration)\n    training_records, validation_records = split_records(records, configuration.data_seed)\n    training_dataset = MaterializedCausalByteDataset(training_records, configuration.sequence_length)\n    validation_dataset = MaterializedCausalByteDataset(validation_records, configuration.sequence_length)\n    dataset_report = {\n        \"records\": len(records),\n        \"training_records\": len(training_records),\n        \"validation_records\": len(validation_records),\n        \"training_chunks\": len(training_dataset),\n        \"validation_chunks\": len(validation_dataset),\n        \"sequence_length\": configuration.sequence_length,\n        \"corpus_manifest\": corpus_manifest,\n    }\n    atomic_write_json(configuration.results_directory / \"dataset_report.json\", dataset_report)\n    session_report = run_training(\n        configuration,\n        corpus_manifest,\n        training_dataset,\n        validation_dataset,\n        device,\n        environment,\n    )\n    final_report = {\"environment\": environment, \"dataset\": dataset_report, \"session\": session_report}\n    atomic_write_json(configuration.results_directory / \"final_report.json\", final_report)\n    print(json.dumps(final_report, indent=2, sort_keys=True))\n\n\nif __name__ == \"__main__\":\n    main()\n"
import sys
import types
MODULE_NAME = 'koemi.training.a100_run_embedded'
MODULE_OBJECT = types.ModuleType(MODULE_NAME)
sys.modules[MODULE_NAME] = MODULE_OBJECT
A100_NAMESPACE = MODULE_OBJECT.__dict__
exec(compile(A100_RUN_SOURCE, 'embedded_koemi_a100_run.py', 'exec'), A100_NAMESPACE)
INTERNAL_TEST_REPORT = A100_NAMESPACE['run_internal_contract_tests']()
print(json.dumps(INTERNAL_TEST_REPORT, indent=2, sort_keys=True))

In [ ]:
RunConfiguration = A100_NAMESPACE['RunConfiguration']
CorpusQuotas = A100_NAMESPACE['CorpusQuotas']
configure_a100 = A100_NAMESPACE['configure_a100']
build_or_load_corpus = A100_NAMESPACE['build_or_load_corpus']
split_records = A100_NAMESPACE['split_records']
MaterializedCausalByteDataset = A100_NAMESPACE['MaterializedCausalByteDataset']
run_training = A100_NAMESPACE['run_training']
atomic_write_json = A100_NAMESPACE['atomic_write_json']

CONFIGURATION = RunConfiguration(
    results_directory=RESULTS_DIR,
    session_seconds=9 * 60 * 60,
    data_seed=20260913,
    model_seed=1337,
    sequence_length=256,
    quotas=CorpusQuotas(
        opencode_priority=60_000,
        opencode_general=120_000,
        codefeedback=80_000,
        magicoder=70_000,
        openr1_math=35_000,
    ),
    opencode_scan_limit=2_000_000,
    source_scan_limit=200_000,
    shuffle_buffer_size=20_000,
    num_workers=2,
    checkpoint_interval_seconds=15 * 60,
    log_interval_steps=10,
    evaluation_batches=128,
)
DEVICE, ENVIRONMENT = configure_a100()
atomic_write_json(RESULTS_DIR / 'environment.json', ENVIRONMENT)
RECORDS, CORPUS_MANIFEST = build_or_load_corpus(CONFIGURATION)
TRAINING_RECORDS, VALIDATION_RECORDS = split_records(RECORDS, CONFIGURATION.data_seed)
TRAINING_DATASET = MaterializedCausalByteDataset(TRAINING_RECORDS, CONFIGURATION.sequence_length)
VALIDATION_DATASET = MaterializedCausalByteDataset(VALIDATION_RECORDS, CONFIGURATION.sequence_length)
DATASET_REPORT = {
    'records': len(RECORDS),
    'training_records': len(TRAINING_RECORDS),
    'validation_records': len(VALIDATION_RECORDS),
    'training_chunks': len(TRAINING_DATASET),
    'validation_chunks': len(VALIDATION_DATASET),
    'sequence_length': CONFIGURATION.sequence_length,
    'corpus_manifest': CORPUS_MANIFEST,
}
atomic_write_json(RESULTS_DIR / 'dataset_report.json', DATASET_REPORT)
print(json.dumps({'environment': ENVIRONMENT, 'dataset': DATASET_REPORT}, indent=2, sort_keys=True))
SESSION_REPORT = run_training(
    CONFIGURATION,
    CORPUS_MANIFEST,
    TRAINING_DATASET,
    VALIDATION_DATASET,
    DEVICE,
    ENVIRONMENT,
)
FINAL_REPORT = {'environment': ENVIRONMENT, 'dataset': DATASET_REPORT, 'session': SESSION_REPORT}
atomic_write_json(RESULTS_DIR / 'final_report.json', FINAL_REPORT)
print(json.dumps(FINAL_REPORT, indent=2, sort_keys=True))

## Inspect after the overnight session

Run this cell after the training cell exits. `final_report.json` contains the environment, corpus hash, selected batch calibration, checkpoint recovery messages, validation loss and peak VRAM. `sample_generation.txt` is a raw-byte sample from the saved model.

In [ ]:
import json

for report_name in ('environment.json', 'internal_contract_tests.json', 'dataset_report.json', 'latest_session.json', 'final_report.json'):
    report_path = RESULTS_DIR / report_name
    if report_path.exists():
        print(f'\n===== {report_name} =====')
        print(report_path.read_text(encoding='utf-8'))
sample_code = RESULTS_DIR / 'sample_generation.txt'
if sample_code.exists():
    print('\n===== sample_generation.txt =====')
    print(sample_code.read_text(encoding='utf-8'))